In [1]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# # Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# # Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# # kagglehub.dataset_download('<owner>/<dataset-slug>')

# Fondation and Data Exploration

## Importing Libraries

In [2]:
import os
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import wandb
from tqdm import tqdm
from kaggle_secrets import UserSecretsClient

## Config

In [3]:
DATA_DIR = "/kaggle/input/competitions/industrial-surface-defect-detection/public"
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TRAIN_IMG_DIR = os.path.join(DATA_DIR, "train")
WANDB_PROJECT = "24f3003188-t32026"
CLASS_NAMES = ['defect_a', 'defect_b', 'defect_c', 'defect_d']
IMG_HEIGHT, IMG_WIDTH = 128, 800

## Wandb Login

In [4]:
try:
    user_secrets = UserSecretsClient()
    wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")
    wandb.login(key=wandb_api_key)
except Exception:
    print("Could not auto-login to WandB. Ensure WANDB_API_KEY is set in Kaggle Secrets.")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


## Some Utitility Functions

In [5]:
def rle2mask(rle_string, height=IMG_HEIGHT, width=IMG_WIDTH):
    """Convert RLE string to binary mask."""
    mask = np.zeros(width * height, dtype=np.uint8)
    if pd.isna(rle_string) or rle_string.strip() == "":
        return mask.reshape((height, width), order='F')
    
    runs = np.array([int(x) for x in rle_string.split()])
    starts = runs[0::2] - 1
    lengths = runs[1::2]
    
    for start, length in zip(starts, lengths):
        mask[start:start+length] = 1
        
    return mask.reshape((height, width), order='F')

def mask2rle(mask):
    """Convert binary mask back to RLE."""
    pixels = mask.T.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return " ".join(str(x) for x in runs)

## Main EDA

In [6]:
def run_eda():
    run = wandb.init(project=WANDB_PROJECT, name="comprehensive_eda", job_type="eda")
    
    print("Loading data...")
    df = pd.read_csv(TRAIN_CSV)

    df_pivot = df.pivot(index='image_id', columns='defect_type', values='mask_rle').reset_index()
    df_pivot.fillna('', inplace=True)
    total_images = len(df_pivot)
    
    for cls in CLASS_NAMES:
        df_pivot[f"has_{cls}"] = (df_pivot[cls] != '').astype(int)
    df_pivot['num_defects'] = df_pivot[[f"has_{cls}" for cls in CLASS_NAMES]].sum(axis=1)

    print(f"Total Unique Images: {total_images}")

    print("Calculating defect distributions...")
    class_counts = {cls: int(df_pivot[f"has_{cls}"].sum()) for cls in CLASS_NAMES}
    defect_freq = df_pivot['num_defects'].value_counts().sort_index().to_dict()

    wandb.log({
        "Class Distribution": wandb.plot.bar(
            wandb.Table(data=[[k, v] for k, v in class_counts.items()], columns=["Defect", "Count"]),
            "Defect", "Count", title="Images per Defect Type"
        ),
        "Defects per Image": wandb.plot.bar(
            wandb.Table(data=[[str(k), v] for k, v in defect_freq.items()], columns=["Number of Defects", "Count"]),
            "Number of Defects", "Count", title="How many defects in a single image?"
        )
    })

    print("Calculating co-occurrence matrix...")
    co_matrix = np.zeros((4, 4), dtype=int)
    for i, cls1 in enumerate(CLASS_NAMES):
        for j, cls2 in enumerate(CLASS_NAMES):
            co_matrix[i, j] = int((df_pivot[f"has_{cls1}"] & df_pivot[f"has_{cls2}"]).sum())
            
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(co_matrix, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title("Defect Co-occurrence Matrix")
    wandb.log({"Co-occurrence Matrix": wandb.Image(fig)})
    plt.close(fig)

    print("Analyzing pixel areas and spatial distribution (this may take a minute)...")
    
    # Store areas for boxplots
    areas = {cls: [] for cls in CLASS_NAMES}
    
    # Accumulate masks for spatial heatmaps
    heatmaps = {cls: np.zeros((IMG_HEIGHT, IMG_WIDTH), dtype=np.float32) for cls in CLASS_NAMES}
    
    # Sample 1000 images for speed, or remove .sample() to do full dataset
    sample_df = df_pivot[df_pivot['num_defects'] > 0].sample(min(1500, len(df_pivot[df_pivot['num_defects'] > 0])), random_state=42)
    
    for _, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
        for cls in CLASS_NAMES:
            if row[cls] != '':
                mask = rle2mask(row[cls])
                area = int(mask.sum())
                areas[cls].append(area)
                heatmaps[cls] += mask

   # 1. Log Area Boxplots using Seaborn
    area_data = []
    for cls in CLASS_NAMES:
        for area in areas[cls]:
            area_data.append({"Defect Class": cls, "Pixel Area": area})
    
    area_df = pd.DataFrame(area_data)
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.boxplot(data=area_df, x="Defect Class", y="Pixel Area", ax=ax, palette="Set2")
    plt.title("Defect Area Size Distribution")
    wandb.log({"Defect Area Size Distribution": wandb.Image(fig)})
    plt.close(fig)

    # 2. Log Spatial Heatmaps (Where do defects typically occur?)
    fig, axes = plt.subplots(4, 1, figsize=(12, 8))
    for i, cls in enumerate(CLASS_NAMES):
        if np.max(heatmaps[cls]) > 0:
            heatmaps[cls] = heatmaps[cls] / np.max(heatmaps[cls])  # normalize
        sns.heatmap(heatmaps[cls], cmap="magma", ax=axes[i], cbar=False, xticklabels=False, yticklabels=False)
        axes[i].set_title(f"Spatial Heatmap: {cls}")
    plt.tight_layout()
    wandb.log({"Spatial Heatmaps": wandb.Image(fig)})
    plt.close(fig)
    
    print("Generating Image Visualizations...")
    
    colors = {'defect_a': (255, 0, 0), 'defect_b': (0, 255, 0), 'defect_c': (0, 0, 255), 'defect_d': (255, 255, 0)}
    
    def log_image_samples(df_subset, log_name, n=5):
        wandb_images = []
        for _, row in df_subset.head(n).iterrows():
            img = cv2.cvtColor(cv2.imread(os.path.join(TRAIN_IMG_DIR, f"{row['image_id']}.jpg")), cv2.COLOR_BGR2RGB)
            overlay = np.zeros_like(img)
            active_defects = []
            
            for cls in CLASS_NAMES:
                if row[cls] != '':
                    active_defects.append(cls)
                    mask = rle2mask(row[cls])
                    overlay[mask == 1] = colors[cls]
            
            blended = cv2.addWeighted(img, 1.0, overlay, 0.5, 0)
            wandb_images.append(wandb.Image(blended, caption=f"ID: {row['image_id']} | Defects: {', '.join(active_defects)}"))
        
        if wandb_images:
            wandb.log({log_name: wandb_images})

    # Log Random Multi-Defect Images
    multi_defect_df = df_pivot[df_pivot['num_defects'] >= 2].sample(10, random_state=42)
    log_image_samples(multi_defect_df, "Multi-Defect Samples", n=10)

    print("EDA Complete! Check WandB Dashboard.")
    wandb.finish()

if __name__ == "__main__":
    run_eda()

wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20261005_074735-2ulwj6sy
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run comprehensive_eda
wandb: ⭐️ View project at https://wandb.ai/atharvsk-indian-institute-of-technology-madras/24f3003188-t32026
wandb: 🚀 View run at https://wandb.ai/atharvsk-indian-institute-of-technology-madras/24f3003188-t32026/runs/2ulwj6sy


Loading data...
Total Unique Images: 11393
Calculating defect distributions...
Calculating co-occurrence matrix...
Analyzing pixel areas and spatial distribution (this may take a minute)...


100%|██████████| 1500/1500 [00:00<00:00, 1933.24it/s]
/tmp/ipykernel_23/1011972550.py:71: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=area_df, x="Defect Class", y="Pixel Area", ax=ax, palette="Set2")


Generating Image Visualizations...


wandb: updating run metadata


EDA Complete! Check WandB Dashboard.


wandb: uploading history steps 0-4, summary, console lines 1-12
wandb: 🚀 View run comprehensive_eda at: https://wandb.ai/atharvsk-indian-institute-of-technology-madras/24f3003188-t32026/runs/2ulwj6sy
wandb: ⭐️ View project at: https://wandb.ai/atharvsk-indian-institute-of-technology-madras/24f3003188-t32026
wandb: Synced 5 W&B file(s), 15 media file(s), 4 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20261005_074735-2ulwj6sy/logs
